In [1]:
"""
TASK 1: Dataset Exploration and Imbalance Analysis
Deep Learning-Based Classification of Imbalanced WCE Datasets
SVNIT Surat | Prof. Praveen Kumar Chandaliya

Covers all 5 datasets:
  1. Kvasir-Capsule     (primary, WCE, 14 classes, heavily imbalanced)
  2. KVASIR v2          (balanced reference, 8 classes)
  3. CVC-ClinicDB       (colonoscopy polyp, binary)
  4. ETIS-Larib         (colonoscopy polyp, binary)
  5. KID                (WCE, 4 classes, cross-validation)

HOW TO RUN:
  python task1_exploration.py

FOLDER STRUCTURE EXPECTED:
  datasets/
  ├── kvasir_capsule/          ← one subfolder per class
  │   ├── Angiectasia/
  │   ├── Blood - fresh/
  │   └── ...
  ├── kvasir_v2/               ← one subfolder per class
  │   ├── dyed-lifted-polyps/
  │   └── ...
  ├── CVC-ClinicDB/
  │   ├── polyp/               ← polyp images
  │   └── non_polyp/           ← normal/background images
  ├── ETIS-Larib/
  │   ├── polyp/
  │   └── non_polyp/
  └── KID/
      ├── angiectasia/
      ├── bleeding/
      ├── normal/
      └── polyps/

IF YOU DON'T HAVE DATASETS YET:
  The script will use hardcoded known statistics and still produce
  all charts and analysis. Set USE_HARDCODED = False below.

DOWNLOAD LINKS:
  Kvasir-Capsule : https://datasets.simula.no/kvasir-capsule/
  KVASIR v2      : https://datasets.simula.no/kvasir/
  CVC-ClinicDB   : https://polyp.grand-challenge.org/CVCClinicDB/
  ETIS-Larib     : https://polyp.grand-challenge.org/ETISLarib/
  KID            : https://mdss.uth.gr/datasets/endoscopy/kid/
"""

import os
import json
import warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from pathlib import Path
from collections import OrderedDict

warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURATION
# ============================================================

# Set to True if you don't have datasets downloaded yet.
# Script will use verified published statistics and still
# produce all charts, analysis, and report.
USE_HARDCODED = False

# Set your dataset root if you have them downloaded
DATASET_ROOT = r"C:\Users\devas\Desktop\DL LAB\Major_project\datasets"

# Output directory for all figures and reports
OUTPUT_DIR = "./task1_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# VERIFIED DATASET STATISTICS (from published papers)
# Sources:
#   Kvasir-Capsule: Smedsrud et al., Scientific Data, 2021
#   KVASIR v2:      Pogorelov et al., MMSys, 2017
#   CVC-ClinicDB:   Bernal et al., Computerized Medical Imaging, 2015
#   ETIS-Larib:     Silva et al., Journal of Healthcare Engineering, 2014
#   KID:            Koulaouzidis et al., Endoscopy International Open, 2017
# ============================================================

HARDCODED_STATS = {

    "Kvasir-Capsule": {
        "description": "Wireless Capsule Endoscopy (WCE). Patient swallows a pill camera. "
                       "Largest public capsule endoscopy dataset. 14 GI findings.",
        "modality": "Wireless Capsule Endoscopy (WCE)",
        "resolution": "336 x 336 px",
        "total_images": 47238,
        "task": "Multi-class Classification",
        "classes": OrderedDict([
            ("Normal clean mucosa",   34338),
            ("Ileocecal valve",        4189),
            ("Reduced mucosal view",   2906),
            ("Pylorus",                1529),
            ("Angiectasia",             866),
            ("Ulcer",                   854),
            ("Foreign body",            776),
            ("Lymphangiectasia",        592),
            ("Erosion",                 506),
            ("Blood - fresh",           446),
            ("Erythema",                159),
            ("Polyp",                    55),
            ("Blood - hematin",          12),
            ("Ampulla of Vater",         10),
        ]),
        "download": "https://datasets.simula.no/kvasir-capsule/"
    },

    "KVASIR v2": {
        "description": "Standard colonoscopy/gastroscopy using a flexible tube. "
                       "Balanced dataset used as a clean comparison benchmark. 8 GI findings.",
        "modality": "Standard Colonoscopy / Gastroscopy",
        "resolution": "720x576 to 1920x1072 px",
        "total_images": 8000,
        "task": "Multi-class Classification",
        "classes": OrderedDict([
            ("Dyed-lifted-polyps",      1000),
            ("Dyed-resection-margins",  1000),
            ("Esophagitis",             1000),
            ("Normal-cecum",            1000),
            ("Normal-pylorus",          1000),
            ("Normal-z-line",           1000),
            ("Polyps",                  1000),
            ("Ulcerative-colitis",      1000),
        ]),
        "download": "https://datasets.simula.no/kvasir/"
    },

    "CVC-ClinicDB": {
        "description": "Colonoscopy frames with polyp segmentation masks. "
                       "Used here as binary classification: polyp vs non-polyp. "
                       "612 unique polyp frames from 31 colonoscopy sequences.",
        "modality": "Standard Colonoscopy",
        "resolution": "574 x 500 px",
        "total_images": 612,
        "task": "Binary Classification (Polyp / Non-Polyp)",
        "classes": OrderedDict([
            ("Polyp",     612),
            ("Non-Polyp", 612),
        ]),
        "download": "https://polyp.grand-challenge.org/CVCClinicDB/"
    },

    "ETIS-Larib": {
        "description": "High-resolution colonoscopy frames for polyp detection. "
                       "Small but high-quality. Often used as test-only benchmark. "
                       "196 polyp frames with expert annotations.",
        "modality": "Standard Colonoscopy",
        "resolution": "1225 x 966 px",
        "total_images": 196,
        "task": "Binary Classification (Polyp / Non-Polyp)",
        "classes": OrderedDict([
            ("Polyp",     196),
            ("Non-Polyp", 196),
        ]),
        "download": "https://polyp.grand-challenge.org/ETISLarib/"
    },

    "KID": {
        "description": "Wireless Capsule Endoscopy dataset from University of Thessaly. "
                       "Same modality as Kvasir-Capsule (swallowed pill camera). "
                       "4 GI pathology classes. Used for cross-dataset validation.",
        "modality": "Wireless Capsule Endoscopy (WCE)",
        "resolution": "360 x 360 px",
        "total_images": 370,
        "task": "Multi-class Classification",
        "classes": OrderedDict([
            ("Normal",       315),
            ("Polyps",        34),
            ("Angiectasia",   18),
            ("Bleeding",       3),
        ]),
        "download": "https://mdss.uth.gr/datasets/endoscopy/kid/"
    },
}

# ============================================================
# UTILITY: LOAD FROM DISK OR USE HARDCODED
# ============================================================

def load_dataset_stats(name, folder_path, hardcoded):
    """
    Try to count images from actual folders.
    Fall back to hardcoded stats if folder not found.
    """
    path = Path(folder_path)

    if path.exists() and not USE_HARDCODED:
        classes = {}
        subdirs = sorted([d for d in path.iterdir() if d.is_dir()])
        if not subdirs:
            print(f"  ⚠  {name}: folder exists but no class subfolders found. Using hardcoded.")
            return hardcoded

        exts = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}
        for subdir in subdirs:
            count = sum(1 for f in subdir.iterdir() if f.suffix.lower() in exts)
            classes[subdir.name] = count

        result = dict(hardcoded)
        result['classes'] = OrderedDict(sorted(classes.items(), key=lambda x: -x[1]))
        result['total_images'] = sum(classes.values())
        print(f"  ✅ {name}: loaded from disk — {result['total_images']} images, "
              f"{len(classes)} classes")
        return result
    else:
        print(f"  📋 {name}: using verified published statistics")
        return hardcoded


def get_all_stats():
    print("\nLoading dataset statistics...")
    print("-" * 50)
    stats = {}
    paths = {
        "Kvasir-Capsule": os.path.join(DATASET_ROOT, "kvasir_capsule"),
        "KVASIR v2":      os.path.join(DATASET_ROOT, "kvasir_v2"),
        "CVC-ClinicDB":   os.path.join(DATASET_ROOT, "CVC-ClinicDB"),
        "ETIS-Larib":     os.path.join(DATASET_ROOT, "ETIS-Larib"),
        "KID":            os.path.join(DATASET_ROOT, "KID"),
    }
    for name, path in paths.items():
        stats[name] = load_dataset_stats(name, path, HARDCODED_STATS[name])
    print()
    return stats


# ============================================================
# ANALYSIS FUNCTIONS
# ============================================================

def compute_imbalance_metrics(classes_dict):
    counts = list(classes_dict.values())
    names  = list(classes_dict.keys())
    total  = sum(counts)
    max_c  = max(counts)
    min_c  = min(counts)
    ratio  = max_c / min_c if min_c > 0 else float('inf')
    percentages = [c / total * 100 for c in counts]
    majority_cls = names[counts.index(max_c)]
    minority_cls = names[counts.index(min_c)]
    return {
        "total": total,
        "num_classes": len(names),
        "max_count": max_c,
        "min_count": min_c,
        "imbalance_ratio": ratio,
        "majority_class": majority_cls,
        "minority_class": minority_cls,
        "percentages": percentages,
        "counts": counts,
        "names": names,
    }


def classify_imbalance(ratio):
    if ratio == 1.0:
        return "Perfectly Balanced", "#27ae60"
    elif ratio < 3:
        return "Mild Imbalance", "#f1c40f"
    elif ratio < 10:
        return "Moderate Imbalance", "#e67e22"
    elif ratio < 100:
        return "Severe Imbalance", "#e74c3c"
    else:
        return "Extreme Imbalance", "#8e1a0e"


# ============================================================
# PLOT 1: Per-Dataset Class Distribution Bar Charts
# ============================================================

def plot_class_distributions(all_stats):
    print("Generating Plot 1: Class distributions per dataset...")

    fig = plt.figure(figsize=(22, 28))
    fig.patch.set_facecolor('#0f0f1a')

    title = fig.suptitle(
        "Task 1 — Class Distribution Across All 5 Datasets\n"
        "WCE Gastrointestinal Disease Classification",
        fontsize=18, fontweight='bold', color='white', y=0.98
    )

    gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.55, wspace=0.35)
    axes_positions = [
        gs[0, 0], gs[0, 1],
        gs[1, 0], gs[1, 1],
        gs[2, 0]
    ]

    dataset_names = list(all_stats.keys())
    palette = [
        '#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6'
    ]

    for ax_idx, (name, pos) in enumerate(zip(dataset_names, axes_positions)):
        ax = fig.add_subplot(pos)
        ax.set_facecolor('#1a1a2e')

        m = compute_imbalance_metrics(all_stats[name]['classes'])
        label, color = classify_imbalance(m['imbalance_ratio'])

        short_names = [n[:22] + '…' if len(n) > 22 else n for n in m['names']]
        x = np.arange(len(short_names))

        bars = ax.bar(
            x, m['counts'],
            color=palette[ax_idx],
            alpha=0.85,
            edgecolor='white',
            linewidth=0.5
        )

        # Annotate bars
        for bar, count in zip(bars, m['counts']):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(m['counts']) * 0.01,
                f'{count:,}',
                ha='center', va='bottom',
                fontsize=7.5, color='white', fontweight='bold'
            )

        ax.set_xticks(x)
        ax.set_xticklabels(short_names, rotation=40, ha='right', fontsize=8.5, color='white')
        ax.set_ylabel('Image Count', color='white', fontsize=9)
        ax.tick_params(axis='y', colors='white', labelsize=8)
        ax.spines[:].set_color('#444')
        ax.set_facecolor('#1a1a2e')

        ax.set_title(
            f"{name}\n"
            f"{m['num_classes']} classes  |  {m['total']:,} images  |  "
            f"{label} (ratio {m['imbalance_ratio']:.0f}:1)",
            color='white', fontsize=10, fontweight='bold', pad=10
        )

        # Imbalance badge
        badge = mpatches.FancyBboxPatch(
            (0.68, 0.88), 0.30, 0.10,
            boxstyle="round,pad=0.01",
            transform=ax.transAxes,
            facecolor=color, alpha=0.85, zorder=5
        )
        ax.add_patch(badge)
        ax.text(
            0.83, 0.93, label,
            transform=ax.transAxes,
            fontsize=7.5, color='white',
            fontweight='bold', ha='center', va='center', zorder=6
        )

    # Summary in last panel
    ax_summary = fig.add_subplot(gs[2, 1])
    ax_summary.set_facecolor('#1a1a2e')
    ax_summary.axis('off')

    summary_lines = [
        ("Dataset", "Classes", "Images", "Imbalance", "Modality"),
    ]
    for name in dataset_names:
        m = compute_imbalance_metrics(all_stats[name]['classes'])
        label, _ = classify_imbalance(m['imbalance_ratio'])
        modality = "WCE" if "WCE" in all_stats[name]['modality'] else "Colonoscopy"
        summary_lines.append((
            name.replace("Kvasir-Capsule", "Kvasir-Cap").replace("KVASIR v2", "KVASIR-v2"),
            str(m['num_classes']),
            f"{m['total']:,}",
            f"{m['imbalance_ratio']:.0f}:1" if m['imbalance_ratio'] != float('inf') else "1:1",
            modality
        ))

    col_x  = [0.0, 0.20, 0.40, 0.58, 0.78]
    row_y  = np.linspace(0.88, 0.10, len(summary_lines))
    colors_row = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6']

    for r_idx, (row, y) in enumerate(zip(summary_lines, row_y)):
        for c_idx, (cell, x) in enumerate(zip(row, col_x)):
            weight = 'bold' if r_idx == 0 else 'normal'
            color  = 'white' if r_idx == 0 else colors_row[r_idx - 1]
            ax_summary.text(
                x, y, cell,
                transform=ax_summary.transAxes,
                fontsize=9, color=color,
                fontweight=weight, va='center'
            )

    ax_summary.set_title(
        "Summary Comparison", color='white',
        fontsize=11, fontweight='bold'
    )

    plt.savefig(
        os.path.join(OUTPUT_DIR, "plot1_class_distributions.png"),
        dpi=150, bbox_inches='tight',
        facecolor='#0f0f1a'
    )
    plt.close()
    print("  ✅ Saved: plot1_class_distributions.png")


# ============================================================
# PLOT 2: Kvasir-Capsule Deep Dive (primary dataset)
# ============================================================

def plot_kvasir_capsule_deep_dive(all_stats):
    print("Generating Plot 2: Kvasir-Capsule deep dive...")

    kc   = all_stats["Kvasir-Capsule"]
    m    = compute_imbalance_metrics(kc['classes'])
    total = m['total']

    fig, axes = plt.subplots(1, 3, figsize=(22, 8))
    fig.patch.set_facecolor('#0f0f1a')
    fig.suptitle(
        "Kvasir-Capsule: Primary Dataset — Imbalance Deep Dive\n"
        "47,238 labelled Wireless Capsule Endoscopy frames | 14 classes",
        fontsize=14, color='white', fontweight='bold'
    )

    for ax in axes:
        ax.set_facecolor('#1a1a2e')
        ax.spines[:].set_color('#444')
        ax.tick_params(colors='white')

    # --- Panel 1: Horizontal bar chart (log scale) ---
    ax = axes[0]
    names   = m['names'][::-1]
    counts  = m['counts'][::-1]
    colors  = plt.cm.RdYlGn(np.linspace(0.1, 0.9, len(names)))[::-1]

    bars = ax.barh(names, counts, color=colors, edgecolor='white', linewidth=0.4)
    for bar, count in zip(bars, counts):
        ax.text(
            count * 1.05, bar.get_y() + bar.get_height() / 2,
            f'{count:,}', va='center', ha='left',
            fontsize=8, color='white'
        )
    ax.set_xscale('log')
    ax.set_xlabel('Image Count (log scale)', color='white', fontsize=9)
    ax.set_title('Class Distribution (log scale)', color='white', fontsize=10, fontweight='bold')
    ax.tick_params(axis='y', labelsize=8, colors='white')
    ax.tick_params(axis='x', colors='white')
    ax.axvline(x=200, color='#e74c3c', linestyle='--', linewidth=1.5, label='Threshold 200')
    ax.legend(fontsize=8, labelcolor='white', facecolor='#1a1a2e', edgecolor='#444')

    # --- Panel 2: Pie chart (top 5 + rest) ---
    ax = axes[1]
    counts_sorted = sorted(zip(m['counts'], m['names']), reverse=True)
    top5 = counts_sorted[:5]
    rest = sum(c for c, _ in counts_sorted[5:])

    pie_vals   = [c for c, _ in top5] + [rest]
    pie_labels = [n[:18] for _, n in top5] + [f"Other\n({len(counts_sorted)-5} classes)"]
    pie_colors = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#3498db', '#95a5a6']
    explode    = [0.05] * len(pie_vals)
    explode[0] = 0.12

    wedges, texts, autotexts = ax.pie(
        pie_vals, labels=pie_labels,
        autopct='%1.1f%%', colors=pie_colors,
        explode=explode, startangle=140,
        textprops={'color': 'white', 'fontsize': 8},
        pctdistance=0.78
    )
    for at in autotexts:
        at.set_fontsize(7.5)
        at.set_color('white')
    ax.set_title(
        'Class Share (Top 5 + Rest)\nNormal clean mucosa = 72.7% of all data',
        color='white', fontsize=10, fontweight='bold'
    )

    # --- Panel 3: Imbalance severity per class ---
    ax = axes[2]
    max_count = max(m['counts'])
    short_names = [n[:18] + '…' if len(n) > 18 else n for n in m['names']]
    imbalance_vs_max = [max_count / c for c in m['counts']]

    cmap_vals = np.array(m['counts']) / max_count
    bar_colors = plt.cm.RdYlGn(cmap_vals)

    x = np.arange(len(short_names))
    bars = ax.bar(x, imbalance_vs_max, color=bar_colors, edgecolor='white', linewidth=0.4)

    for bar, val in zip(bars, imbalance_vs_max):
        if val > 100:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 50,
                f'{val:.0f}x',
                ha='center', va='bottom', fontsize=7, color='#e74c3c', fontweight='bold'
            )

    ax.set_xticks(x)
    ax.set_xticklabels(short_names, rotation=45, ha='right', fontsize=7.5, color='white')
    ax.set_ylabel('× more images than this class', color='white', fontsize=9)
    ax.set_title(
        'Imbalance Severity\n(How many times majority > each class)',
        color='white', fontsize=10, fontweight='bold'
    )
    ax.tick_params(axis='y', colors='white')
    ax.set_yscale('log')

    plt.tight_layout()
    plt.savefig(
        os.path.join(OUTPUT_DIR, "plot2_kvasir_capsule_deep_dive.png"),
        dpi=150, bbox_inches='tight', facecolor='#0f0f1a'
    )
    plt.close()
    print("  ✅ Saved: plot2_kvasir_capsule_deep_dive.png")


# ============================================================
# PLOT 3: Cross-Dataset Comparison
# ============================================================

def plot_cross_dataset_comparison(all_stats):
    print("Generating Plot 3: Cross-dataset comparison...")

    fig, axes = plt.subplots(1, 3, figsize=(20, 7))
    fig.patch.set_facecolor('#0f0f1a')
    fig.suptitle(
        "Cross-Dataset Comparison — All 5 Datasets",
        fontsize=14, color='white', fontweight='bold'
    )
    for ax in axes:
        ax.set_facecolor('#1a1a2e')
        ax.spines[:].set_color('#444')

    names   = list(all_stats.keys())
    totals  = [all_stats[n]['total_images'] for n in names]
    n_cls   = [len(all_stats[n]['classes']) for n in names]
    metrics = [compute_imbalance_metrics(all_stats[n]['classes']) for n in names]
    ratios  = [m['imbalance_ratio'] for m in metrics]

    palette = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6']

    # Panel 1: Total images
    ax = axes[0]
    bars = ax.bar(names, totals, color=palette, edgecolor='white', linewidth=0.5)
    for bar, val in zip(bars, totals):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() * 1.02,
            f'{val:,}',
            ha='center', va='bottom', fontsize=9, color='white', fontweight='bold'
        )
    ax.set_title('Total Images per Dataset', color='white', fontsize=11, fontweight='bold')
    ax.set_ylabel('Image Count', color='white')
    ax.tick_params(colors='white', labelsize=8)
    ax.set_xticklabels(names, rotation=20, ha='right', fontsize=8, color='white')
    ax.set_yscale('log')

    # Panel 2: Number of classes
    ax = axes[1]
    bars = ax.bar(names, n_cls, color=palette, edgecolor='white', linewidth=0.5)
    for bar, val in zip(bars, n_cls):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.1,
            str(val),
            ha='center', va='bottom', fontsize=11, color='white', fontweight='bold'
        )
    ax.set_title('Number of Classes', color='white', fontsize=11, fontweight='bold')
    ax.set_ylabel('Class Count', color='white')
    ax.tick_params(colors='white', labelsize=8)
    ax.set_xticklabels(names, rotation=20, ha='right', fontsize=8, color='white')

    # Panel 3: Imbalance ratio
    ax = axes[2]
    ratio_colors = []
    for r in ratios:
        _, c = classify_imbalance(r)
        ratio_colors.append(c)
    bars = ax.bar(names, ratios, color=ratio_colors, edgecolor='white', linewidth=0.5)
    for bar, val in zip(bars, ratios):
        label = f'{val:.0f}:1' if val != 1.0 else '1:1'
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() * 1.05,
            label,
            ha='center', va='bottom', fontsize=9, color='white', fontweight='bold'
        )
    ax.set_title('Imbalance Ratio (Majority : Minority)', color='white', fontsize=11, fontweight='bold')
    ax.set_ylabel('Ratio', color='white')
    ax.set_yscale('log')
    ax.tick_params(colors='white', labelsize=8)
    ax.set_xticklabels(names, rotation=20, ha='right', fontsize=8, color='white')

    # Legend for imbalance colors
    legend_elements = [
        mpatches.Patch(facecolor='#27ae60', label='Perfectly Balanced'),
        mpatches.Patch(facecolor='#f1c40f', label='Mild (<3:1)'),
        mpatches.Patch(facecolor='#e67e22', label='Moderate (<10:1)'),
        mpatches.Patch(facecolor='#e74c3c', label='Severe (<100:1)'),
        mpatches.Patch(facecolor='#8e1a0e', label='Extreme (>100:1)'),
    ]
    axes[2].legend(
        handles=legend_elements,
        loc='upper left', fontsize=8,
        facecolor='#1a1a2e', edgecolor='#444',
        labelcolor='white'
    )

    plt.tight_layout()
    plt.savefig(
        os.path.join(OUTPUT_DIR, "plot3_cross_dataset_comparison.png"),
        dpi=150, bbox_inches='tight', facecolor='#0f0f1a'
    )
    plt.close()
    print("  ✅ Saved: plot3_cross_dataset_comparison.png")


# ============================================================
# PLOT 4: Modality Comparison (WCE vs Colonoscopy)
# ============================================================

def plot_modality_comparison(all_stats):
    print("Generating Plot 4: WCE vs Colonoscopy modality comparison...")

    fig, ax = plt.subplots(figsize=(14, 7))
    fig.patch.set_facecolor('#0f0f1a')
    ax.set_facecolor('#1a1a2e')
    ax.spines[:].set_color('#444')

    wce_datasets = {
        "Kvasir-Capsule\n(14 classes)": all_stats["Kvasir-Capsule"]["total_images"],
        "KID\n(4 classes)":             all_stats["KID"]["total_images"],
    }
    colonoscopy_datasets = {
        "KVASIR v2\n(8 classes)":     all_stats["KVASIR v2"]["total_images"],
        "CVC-ClinicDB\n(2 classes)":  all_stats["CVC-ClinicDB"]["total_images"],
        "ETIS-Larib\n(2 classes)":    all_stats["ETIS-Larib"]["total_images"],
    }

    all_labels = list(wce_datasets.keys()) + [""] + list(colonoscopy_datasets.keys())
    all_values = list(wce_datasets.values()) + [0] + list(colonoscopy_datasets.values())
    colors = (
        ['#3498db', '#9b59b6'] +
        ['none'] +
        ['#2ecc71', '#e74c3c', '#f39c12']
    )

    x = np.arange(len(all_labels))
    bars = ax.bar(x, all_values, color=colors, edgecolor='white', linewidth=0.6, width=0.6)

    for bar, val in zip(bars, all_values):
        if val > 0:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height() * 1.03,
                f'{val:,}',
                ha='center', va='bottom', fontsize=10, color='white', fontweight='bold'
            )

    ax.axvline(x=2.5, color='white', linestyle='--', linewidth=1.2, alpha=0.5)
    ax.text(0.8, ax.get_ylim()[1] * 0.92,
            '← WCE (Capsule Camera)', color='#3498db',
            fontsize=11, fontweight='bold')
    ax.text(3.0, ax.get_ylim()[1] * 0.92,
            'Colonoscopy (Standard Scope) →', color='#2ecc71',
            fontsize=11, fontweight='bold')

    ax.set_xticks(x)
    ax.set_xticklabels(all_labels, fontsize=10, color='white')
    ax.set_ylabel('Total Images', color='white', fontsize=11)
    ax.tick_params(colors='white')
    ax.set_title(
        'Modality Comparison: WCE (Capsule) vs Standard Colonoscopy\n'
        'Both types included — model must generalize across modalities',
        color='white', fontsize=12, fontweight='bold'
    )

    total_wce  = sum(wce_datasets.values())
    total_col  = sum(colonoscopy_datasets.values())
    ax.text(
        0.02, 0.75,
        f"Total WCE images:         {total_wce:,}\nTotal Colonoscopy images: {total_col:,}",
        transform=ax.transAxes,
        fontsize=10, color='white',
        bbox=dict(facecolor='#0f0f1a', edgecolor='#444', alpha=0.8, pad=8)
    )

    plt.tight_layout()
    plt.savefig(
        os.path.join(OUTPUT_DIR, "plot4_modality_comparison.png"),
        dpi=150, bbox_inches='tight', facecolor='#0f0f1a'
    )
    plt.close()
    print("  ✅ Saved: plot4_modality_comparison.png")


# ============================================================
# PRINT TEXT REPORT
# ============================================================

def print_full_report(all_stats):
    sep = "=" * 70

    print(f"\n{sep}")
    print("  TASK 1 — DATASET EXPLORATION REPORT")
    print(f"  Deep Learning-Based WCE Classification | SVNIT Surat")
    print(sep)

    total_all = sum(s['total_images'] for s in all_stats.values())
    total_cls = sum(len(s['classes']) for s in all_stats.values())

    print(f"\n  OVERVIEW")
    print(f"  Datasets analysed : {len(all_stats)}")
    print(f"  Total images      : {total_all:,}")
    print(f"  Total classes     : {total_cls} (across all datasets)")
    print(f"  Modalities        : WCE (capsule camera) + Standard Colonoscopy\n")

    for name, stats in all_stats.items():
        m     = compute_imbalance_metrics(stats['classes'])
        label, _ = classify_imbalance(m['imbalance_ratio'])
        print(f"\n  {'─'*66}")
        print(f"  {name.upper()}")
        print(f"  {'─'*66}")
        print(f"  Modality       : {stats['modality']}")
        print(f"  Task           : {stats['task']}")
        print(f"  Resolution     : {stats['resolution']}")
        print(f"  Total images   : {m['total']:,}")
        print(f"  Num classes    : {m['num_classes']}")
        print(f"  Majority class : {m['majority_class']}  ({m['max_count']:,} images)")
        print(f"  Minority class : {m['minority_class']}  ({m['min_count']:,} images)")
        print(f"  Imbalance ratio: {m['imbalance_ratio']:.0f}:1  →  {label}")
        print(f"  Download       : {stats['download']}")
        print(f"\n  Class breakdown:")
        print(f"  {'Class':<30} {'Count':>8} {'%':>7}  Visual")
        print(f"  {'─'*58}")
        for cls, count in stats['classes'].items():
            pct   = count / m['total'] * 100
            bar   = '█' * int(pct / 2) + '░' * (50 - int(pct / 2))
            flag  = " ⚠ MINORITY" if count < 100 else ""
            print(f"  {cls:<30} {count:>8,} {pct:>6.2f}%  {bar[:20]}{flag}")

    print(f"\n{sep}")
    print("  WHY CLASS IMBALANCE IS CRITICAL IN MEDICAL DIAGNOSIS")
    print(sep)
    print("""
  In medical image classification, class imbalance is far more dangerous
  than in general computer vision tasks. In Kvasir-Capsule, 'Normal clean
  mucosa' accounts for 72.7% of all data, while 'Ampulla of Vater' has only
  10 samples. A model trained naively on this data will achieve ~73% accuracy
  by simply predicting 'normal' for every frame — yet completely miss every
  actual pathology. In clinical practice, missing a polyp (55 samples) or
  fresh bleeding (446 samples) is not a statistical error — it is a missed
  cancer or a life-threatening haemorrhage. The minority classes represent
  exactly the rare but critical findings that AI must detect. Standard
  accuracy metrics are therefore meaningless here; per-class recall, F1-score,
  and confusion matrices are essential for evaluating real diagnostic utility.
  """)

    print(sep)
    print("  SAVED OUTPUTS")
    print(sep)
    for f in sorted(os.listdir(OUTPUT_DIR)):
        print(f"  {os.path.join(OUTPUT_DIR, f)}")
    print()


# ============================================================
# SAVE JSON REPORT
# ============================================================

def save_json_report(all_stats):
    report = {}
    for name, stats in all_stats.items():
        m = compute_imbalance_metrics(stats['classes'])
        label, _ = classify_imbalance(m['imbalance_ratio'])
        report[name] = {
            "modality":        stats['modality'],
            "task":            stats['task'],
            "total_images":    m['total'],
            "num_classes":     m['num_classes'],
            "imbalance_ratio": round(m['imbalance_ratio'], 2),
            "imbalance_level": label,
            "majority_class":  m['majority_class'],
            "minority_class":  m['minority_class'],
            "class_counts":    dict(stats['classes']),
            "download":        stats['download'],
        }

    path = os.path.join(OUTPUT_DIR, "task1_report.json")
    with open(path, 'w') as f:
        json.dump(report, f, indent=2)
    print(f"  ✅ Saved: task1_report.json")


# ============================================================
# MAIN
# ============================================================

def main():
    print("\n" + "="*70)
    print("  TASK 1: Dataset Exploration and Imbalance Analysis")
    print("  WCE Gastrointestinal Classification | SVNIT Surat")
    print("="*70)

    # Load or use hardcoded stats
    all_stats = get_all_stats()

    # Generate all plots
    print("\nGenerating visualisations...")
    print("-" * 50)
    plot_class_distributions(all_stats)
    plot_kvasir_capsule_deep_dive(all_stats)
    plot_cross_dataset_comparison(all_stats)
    plot_modality_comparison(all_stats)

    # Print and save reports
    print_full_report(all_stats)
    save_json_report(all_stats)

    print("="*70)
    print("  TASK 1 COMPLETE")
    print(f"  All outputs saved to: {OUTPUT_DIR}/")
    print("="*70 + "\n")


if __name__ == '__main__':
    main()


  TASK 1: Dataset Exploration and Imbalance Analysis
  WCE Gastrointestinal Classification | SVNIT Surat

Loading dataset statistics...
--------------------------------------------------
  ✅ Kvasir-Capsule: loaded from disk — 47238 images, 14 classes
  ✅ KVASIR v2: loaded from disk — 8000 images, 8 classes
  ✅ CVC-ClinicDB: loaded from disk — 612 images, 1 classes
  ✅ ETIS-Larib: loaded from disk — 196 images, 1 classes
  ✅ KID: loaded from disk — 370 images, 4 classes


Generating visualisations...
--------------------------------------------------
Generating Plot 1: Class distributions per dataset...
  ✅ Saved: plot1_class_distributions.png
Generating Plot 2: Kvasir-Capsule deep dive...
  ✅ Saved: plot2_kvasir_capsule_deep_dive.png
Generating Plot 3: Cross-dataset comparison...
  ✅ Saved: plot3_cross_dataset_comparison.png
Generating Plot 4: WCE vs Colonoscopy modality comparison...
  ✅ Saved: plot4_modality_comparison.png

  TASK 1 — DATASET EXPLORATION REPORT
  Deep Learning-Based

In [2]:
"""
DATASET EXPLANATION: Wireless Capsule Endoscopy (WCE) Datasets
Deep Learning-Based Classification of Imbalanced WCE Datasets
SVNIT Surat | Prof. Praveen Kumar Chandaliya

This script explains all 5 datasets in full detail:
  - What the dataset is
  - How images are captured (medical context)
  - All classes and what they mean medically
  - Key features and characteristics
  - Why each dataset matters for this project

Run:
  python dataset_explanation.py
"""

# ============================================================
# COLOR CODES FOR TERMINAL OUTPUT
# ============================================================

class C:
    RESET   = '\033[0m'
    BOLD    = '\033[1m'
    UNDER   = '\033[4m'

    # Text colors
    WHITE   = '\033[97m'
    CYAN    = '\033[96m'
    GREEN   = '\033[92m'
    YELLOW  = '\033[93m'
    RED     = '\033[91m'
    MAGENTA = '\033[95m'
    BLUE    = '\033[94m'
    ORANGE  = '\033[33m'
    GRAY    = '\033[90m'

def h1(text):
    """Big section header"""
    w = 72
    print(f"\n{C.CYAN}{C.BOLD}{'═' * w}{C.RESET}")
    print(f"{C.CYAN}{C.BOLD}  {text}{C.RESET}")
    print(f"{C.CYAN}{C.BOLD}{'═' * w}{C.RESET}")

def h2(text):
    """Sub-section header"""
    print(f"\n{C.YELLOW}{C.BOLD}  ┌─ {text} {'─' * max(0, 62 - len(text))}┐{C.RESET}")

def h3(text):
    """Smaller header"""
    print(f"\n{C.GREEN}{C.BOLD}  ◆ {text}{C.RESET}")

def body(text, indent=4):
    """Regular body text"""
    prefix = ' ' * indent
    for line in text.strip().split('\n'):
        print(f"{C.WHITE}{prefix}{line.strip()}{C.RESET}")

def bullet(items, indent=6, color=C.WHITE):
    """Bullet point list"""
    for item in items:
        print(f"{color}{' ' * indent}•  {item}{C.RESET}")

def field(label, value, indent=6):
    """Key: Value field"""
    print(f"{' ' * indent}{C.CYAN}{C.BOLD}{label:<22}{C.RESET}{C.WHITE}{value}{C.RESET}")

def warn(text, indent=6):
    """Warning / important note"""
    print(f"{' ' * indent}{C.RED}{C.BOLD}⚠  {text}{C.RESET}")

def good(text, indent=6):
    """Positive note"""
    print(f"{' ' * indent}{C.GREEN}✓  {text}{C.RESET}")

def info(text, indent=6):
    """Info note"""
    print(f"{' ' * indent}{C.BLUE}ℹ  {text}{C.RESET}")

def class_row(name, count, total, note='', is_minority=False):
    """Formatted class entry"""
    pct      = count / total * 100
    bar_len  = int(pct / 2.5)
    bar      = '█' * bar_len + '░' * (28 - bar_len)
    color    = C.RED if is_minority else (C.YELLOW if pct < 5 else C.GREEN)
    flag     = f"  {C.RED}{C.BOLD}← MINORITY ⚠{C.RESET}" if is_minority else ""
    print(f"      {color}{name:<30}{C.WHITE}{count:>7,}  {pct:>5.1f}%  {bar}{flag}")
    if note:
        print(f"      {C.GRAY}{'':30}  {note}{C.RESET}")


# ============================================================
# SECTION 0: OVERVIEW
# ============================================================

def print_overview():
    h1("DATASET OVERVIEW — WCE Gastrointestinal Classification")

    body("""
This project works with 5 medical imaging datasets for gastrointestinal (GI)
disease classification. Together they cover two distinct imaging modalities:

  (1) Wireless Capsule Endoscopy (WCE) — patient swallows a pill-sized camera
  (2) Standard Colonoscopy — a flexible tube inserted by a gastroenterologist

Understanding both modalities is essential because the project requires a
single model that generalises across all 5 datasets.
    """)

    h3("The 5 Datasets At A Glance")
    print()
    print(f"      {C.CYAN}{C.BOLD}{'Dataset':<22} {'Modality':<22} {'Classes':>8} {'Images':>10} {'Imbalance':<18}{C.RESET}")
    print(f"      {'─'*80}")

    rows = [
        ("Kvasir-Capsule",  "WCE (Capsule)",         "14",  "47,238",  "EXTREME  3434:1",  C.RED),
        ("KVASIR v2",       "Standard Endoscopy",    " 8",   "8,000",  "BALANCED    1:1",  C.GREEN),
        ("CVC-ClinicDB",    "Colonoscopy",           " 2",     "612",  "BALANCED    1:1",  C.GREEN),
        ("ETIS-Larib",      "Colonoscopy",           " 2",     "196",  "BALANCED    1:1",  C.GREEN),
        ("KID",             "WCE (Capsule)",         " 4",     "370",  "MODERATE  10:1",   C.YELLOW),
    ]
    for name, mod, cls, imgs, imb, col in rows:
        print(f"      {C.WHITE}{name:<22} {mod:<22} {cls:>8} {imgs:>10}  {col}{imb}{C.RESET}")

    print()
    info("Total images across all 5 datasets: 56,416")
    info("Total unique GI classes across datasets: 28 (with some overlap)")
    warn("Kvasir-Capsule dominates — it alone is 80.9% of all data")


# ============================================================
# SECTION 1: WHAT IS WIRELESS CAPSULE ENDOSCOPY?
# ============================================================

def print_wce_explanation():
    h1("WHAT IS WIRELESS CAPSULE ENDOSCOPY (WCE)?")

    body("""
Wireless Capsule Endoscopy is a non-invasive medical imaging technique invented
in 2000. It is the only method that can visually examine the entire small
intestine — a 3 to 5 metre organ that a standard endoscope cannot reach.
    """)

    h3("How It Works — Step by Step")
    bullet([
        "The patient swallows a capsule (26mm × 11mm) containing a camera, light, battery and transmitter",
        "The capsule travels passively through the GI tract — pushed by natural muscle contractions",
        "It captures images at 2 to 6 frames per second continuously for 8 to 12 hours",
        "Signals are transmitted wirelessly to a recorder worn on the patient's belt",
        "The capsule is naturally excreted and is disposable",
        "A single examination produces approximately 50,000 to 60,000 image frames",
        "A gastroenterologist must manually review all frames — takes 1 to 2 hours per patient",
    ])

    h3("Why AI Is Critically Needed Here")
    bullet([
        "Reviewing 50,000 frames manually per patient is exhausting and error-prone",
        "Critical findings (polyps, bleeds) may occupy only 0.1% of all frames",
        "Missed pathologies in capsule endoscopy are a significant clinical problem",
        "AI can pre-screen frames and flag suspicious regions for doctor review",
        "This reduces review time from hours to minutes",
    ], color=C.GREEN)

    h3("WCE vs Standard Endoscopy — Key Differences")
    print()
    print(f"      {C.CYAN}{C.BOLD}{'Property':<30} {'WCE (Capsule)':<25} {'Standard Endoscopy'}{C.RESET}")
    print(f"      {'─' * 70}")
    comparisons = [
        ("Control",             "Passive (no control)",   "Doctor-controlled"),
        ("Area covered",        "Full small intestine",   "Stomach + large bowel"),
        ("Invasion level",      "Non-invasive",           "Invasive (sedation needed)"),
        ("Patient discomfort",  "None (just swallowing)", "Moderate discomfort"),
        ("Frames per exam",     "~50,000 frames",         "~200-500 key frames"),
        ("Image quality",       "336×336, lower SNR",     "Higher resolution"),
        ("Biopsy possible?",    "No",                     "Yes"),
        ("AI difficulty",       "Very High (imbalanced)", "Moderate"),
    ]
    for prop, wce, std in comparisons:
        print(f"      {C.WHITE}{prop:<30} {C.BLUE}{wce:<25} {C.GREEN}{std}{C.RESET}")


# ============================================================
# SECTION 2: KVASIR-CAPSULE
# ============================================================

def print_kvasir_capsule():
    h1("DATASET 1: KVASIR-CAPSULE")

    h2("What Is This Dataset?")
    body("""
Kvasir-Capsule is the largest publicly available annotated Wireless Capsule
Endoscopy dataset in the world. It was published in 2021 by Smedsrud et al.
from Simula Research Laboratory and Oslo University Hospital, Norway.

The dataset was created by reviewing actual patient capsule endoscopy recordings,
extracting key frames, and having expert gastroenterologists annotate each image
with the correct GI finding. It is the PRIMARY dataset for this project.
    """)

    h2("Technical Specifications")
    print()
    field("Source:",           "Simula Research Laboratory, Norway (2021)")
    field("Modality:",         "Wireless Capsule Endoscopy (WCE)")
    field("Image resolution:", "336 × 336 pixels (native capsule resolution)")
    field("Color space:",      "RGB")
    field("Total images:",     "47,238 labelled frames")
    field("Total videos:",     "117 raw capsule videos (4.7 million frames)")
    field("Labelled frames:",  "47,238 out of 4,741,504 (1% labelled)")
    field("Annotation:",       "Expert gastroenterologist verified")
    field("Download:",         "https://datasets.simula.no/kvasir-capsule/")

    h2("All 14 Classes — With Medical Explanation")
    print()
    print(f"      {C.CYAN}{C.BOLD}{'Class Name':<32} {'Count':>7}  {'%':>5}  {'Visual Bar':<30}  Note{C.RESET}")
    print(f"      {'─' * 90}")

    TOTAL = 47238
    classes = [
        ("Normal clean mucosa",    34338, False,
         "Healthy small intestinal lining. Pink, smooth, uniform villi."),
        ("Ileocecal valve",         4189, False,
         "Junction between small and large intestine. Normal landmark."),
        ("Reduced mucosal view",    2906, False,
         "Image obscured by bubbles, debris, or fluid. Not a disease."),
        ("Pylorus",                 1529, False,
         "Gate between stomach and small intestine. Normal landmark."),
        ("Angiectasia",              866, False,
         "Dilated, fragile blood vessels in the gut wall. Can bleed."),
        ("Ulcer",                    854, False,
         "Open sore in the intestinal lining. Painful, may bleed."),
        ("Foreign body",             776, False,
         "Swallowed objects (pills, seeds, wires). Not a GI disease."),
        ("Lymphangiectasia",         592, False,
         "Dilated lymph vessels causing protein loss. Rare condition."),
        ("Erosion",                  506, False,
         "Shallow mucosal damage, less deep than ulcer. Early disease."),
        ("Blood - fresh",            446, False,
         "Active gastrointestinal bleeding. Urgent clinical finding."),
        ("Erythema",                 159, False,
         "Redness/inflammation of the mucosa. Early inflammatory sign."),
        ("Polyp",                     55, True,
         "Abnormal tissue growth. Critical — can become cancerous."),
        ("Blood - hematin",           12, True,
         "Old/digested blood (dark). Indicates prior bleeding event."),
        ("Ampulla of Vater",          10, True,
         "Opening of bile/pancreatic duct. Rarely seen in WCE."),
    ]

    for name, count, minority, note in classes:
        class_row(name, count, TOTAL, note, minority)

    print()
    h2("Key Dataset Characteristics")

    h3("Extreme Class Imbalance")
    body("""
This is the most severely imbalanced medical imaging dataset in common use.
The imbalance ratio of 3,434:1 (Normal mucosa vs Ampulla of Vater) means
that for every 1 image of the rarest class, there are 3,434 normal images.
    """)
    warn("A naive model predicting 'Normal' for everything gets 72.69% accuracy")
    warn("Yet it detects ZERO pathologies — clinically useless")
    good("This is exactly why imbalance handling is the core challenge here")

    h3("Why the Minority Classes Matter Most Clinically")
    bullet([
        "Polyp (55 images)  — can develop into colorectal cancer if missed",
        "Blood-fresh (446)  — active bleeding requires immediate intervention",
        "Ulcer (854)        — indicates inflammatory bowel disease or infection",
        "Angiectasia (866)  — recurrent bleeding source, needs treatment",
        "Erythema (159)     — early sign of Crohn's disease or celiac disease",
    ], color=C.RED)

    h3("What Makes It Difficult for Models")
    bullet([
        "Many pathologies are visually subtle — small lesions in large frames",
        "Color and texture vary between patients and lighting conditions",
        "Motion blur is common due to passive capsule movement",
        "Overlapping visual appearance between some disease classes",
        "The 336×336 resolution is lower than standard endoscopy",
    ])


# ============================================================
# SECTION 3: KVASIR V2
# ============================================================

def print_kvasir_v2():
    h1("DATASET 2: KVASIR v2")

    h2("What Is This Dataset?")
    body("""
KVASIR v2 is a multi-class gastrointestinal endoscopy dataset from the same
research group that created Kvasir-Capsule. However, this dataset uses standard
flexible endoscopy (not capsule) and is deliberately designed to be perfectly
balanced — 1,000 images per class.

Its role in this project is as a COMPARISON BENCHMARK. Because it is balanced,
it lets you evaluate whether your model handles balanced vs imbalanced data
differently, and whether the imbalance techniques you apply to Kvasir-Capsule
actually help.
    """)

    h2("Technical Specifications")
    print()
    field("Source:",           "Simula Research Laboratory, Norway (2017)")
    field("Modality:",         "Standard Flexible Endoscopy (gastroscope/colonoscope)")
    field("Image resolution:", "720×576 to 1920×1072 pixels (variable)")
    field("Color space:",      "RGB")
    field("Total images:",     "8,000 (exactly 1,000 per class)")
    field("Annotation:",       "Expert gastroenterologist verified")
    field("Imbalance ratio:",  "1:1 — Perfectly Balanced")
    field("Download:",         "https://datasets.simula.no/kvasir/")

    h2("All 8 Classes — With Medical Explanation")
    print()
    TOTAL = 8000
    classes = [
        ("Dyed-lifted-polyps",     1000,
         "Polyp injected with dye and lifted before removal. Post-treatment."),
        ("Dyed-resection-margins", 1000,
         "Tissue margin after polyp removal, marked with dye."),
        ("Esophagitis",            1000,
         "Inflammation of the oesophagus. Often due to acid reflux."),
        ("Normal-cecum",           1000,
         "Healthy tissue at the start of the large intestine."),
        ("Normal-pylorus",         1000,
         "Healthy pyloric valve between stomach and small intestine."),
        ("Normal-z-line",          1000,
         "Junction between oesophagus and stomach. Normal landmark."),
        ("Polyps",                 1000,
         "Abnormal tissue growths. Pre-cancerous if adenomatous."),
        ("Ulcerative-colitis",     1000,
         "Chronic inflammatory bowel disease affecting the colon."),
    ]
    print(f"      {C.CYAN}{C.BOLD}{'Class Name':<30} {'Count':>7}  {'%':>5}  {'Medical Note'}{C.RESET}")
    print(f"      {'─' * 80}")
    for name, count, note in classes:
        pct = count / TOTAL * 100
        print(f"      {C.GREEN}{name:<30}{C.WHITE}{count:>7,}  {pct:>5.1f}%  {C.GRAY}{note}{C.RESET}")

    h2("Key Characteristics")
    h3("Why This Dataset Is Useful Here")
    bullet([
        "Perfect balance (1,000 per class) gives a clean training baseline",
        "Shows what model performance looks like WITHOUT imbalance noise",
        "Standard endoscopy images are higher resolution than WCE",
        "Some classes overlap with Kvasir-Capsule (polyps, ulcers) — useful for transfer",
        "Commonly used benchmark — results are directly comparable to literature",
    ], color=C.GREEN)

    h3("Key Difference From Kvasir-Capsule")
    bullet([
        "This is NOT capsule endoscopy — doctor controls the camera",
        "Images are much higher quality and resolution",
        "Fewer frames per examination — doctor captures key moments",
        "No passive motion blur or lighting inconsistencies from capsule travel",
    ])


# ============================================================
# SECTION 4: CVC-CLINICDB
# ============================================================

def print_cvc_clinicdb():
    h1("DATASET 3: CVC-ClinicDB")

    h2("What Is This Dataset?")
    body("""
CVC-ClinicDB was created by the Computer Vision Center (CVC) in Barcelona,
Spain. It is a colonoscopy dataset originally designed for POLYP SEGMENTATION
(pixel-level annotation), but in this project we use it for binary
CLASSIFICATION: does this frame contain a polyp or not?

This dataset is important because polyp detection is one of the most
clinically critical tasks in GI endoscopy — polyps missed during colonoscopy
are a leading cause of interval colorectal cancer.
    """)

    h2("Technical Specifications")
    print()
    field("Source:",           "Computer Vision Center (CVC), Barcelona (2015)")
    field("Modality:",         "Standard Colonoscopy (flexible scope)")
    field("Image resolution:", "574 × 500 pixels")
    field("Color space:",      "RGB")
    field("Total frames:",     "612 polyp frames from 31 colonoscopy sequences")
    field("Annotation type:",  "Pixel-level segmentation masks + classification label")
    field("Task (our use):",   "Binary classification — Polyp vs Non-Polyp")
    field("Imbalance ratio:",  "1:1 (we create equal non-polyp samples)")
    field("Download:",         "https://polyp.grand-challenge.org/CVCClinicDB/")

    h2("Classes")
    print()
    bullet([
        "Polyp       — frame contains at least one visible polyp (612 images)",
        "Non-Polyp   — normal healthy colon mucosa (612 matched images)",
    ], color=C.WHITE)

    h2("Key Characteristics")
    h3("What Makes This Dataset Distinctive")
    bullet([
        "Every image has a precisely annotated polyp mask — pixel-perfect labels",
        "Images come from 31 real clinical colonoscopy sequences",
        "High-quality frames — selected specifically to show polyps clearly",
        "Polyps vary in size, shape, colour, and texture across frames",
        "Standard benchmark for polyp detection — widely cited in literature",
    ], color=C.GREEN)

    h3("Why It Is Small (Only 612 Images)")
    body("""
Annotating polyp frames at pixel level requires expert gastroenterologist time.
Each annotation can take 15-30 minutes per image. This makes large-scale
annotated colonoscopy datasets extremely expensive to create. The 612 images
in CVC-ClinicDB represent significant expert effort.
    """)

    warn("Only 612 images — too small to train from scratch, only fine-tuning works")
    info("Commonly used as a TEST SET in literature, not for training")


# ============================================================
# SECTION 5: ETIS-LARIB
# ============================================================

def print_etis_larib():
    h1("DATASET 4: ETIS-Larib Polyp DB")

    h2("What Is This Dataset?")
    body("""
ETIS-Larib was created by researchers from ETIS Lab (France) and Larib
Hospital. It is a high-resolution colonoscopy dataset specifically designed
for challenging polyp detection scenarios — images were selected to include
DIFFICULT polyps that are easy to miss: flat, small, or camouflaged polyps.

It is widely used in the literature as a HOLD-OUT TEST SET — meaning models
are trained on other data and then evaluated on ETIS-Larib to test
generalisation to hard cases.
    """)

    h2("Technical Specifications")
    print()
    field("Source:",           "ETIS Lab (France) + Larib Hospital (2014)")
    field("Modality:",         "Standard Colonoscopy (flexible scope)")
    field("Image resolution:", "1225 × 966 pixels (high resolution)")
    field("Color space:",      "RGB")
    field("Total frames:",     "196 polyp frames")
    field("Annotation type:",  "Pixel-level segmentation masks")
    field("Task (our use):",   "Binary classification — Polyp vs Non-Polyp")
    field("Imbalance ratio:",  "1:1 (we match with equal non-polyp samples)")
    field("Download:",         "https://polyp.grand-challenge.org/ETISLarib/")

    h2("Classes")
    print()
    bullet([
        "Polyp       — frame with at least one annotated polyp (196 images)",
        "Non-Polyp   — normal colonoscopy mucosa (196 matched images)",
    ], color=C.WHITE)

    h2("Key Characteristics")
    h3("Why This Dataset Is Uniquely Challenging")
    bullet([
        "Images were specifically selected for DIFFICULT polyps — not easy cases",
        "Includes flat polyps (sessile) that are nearly invisible in colour",
        "Includes diminutive polyps (< 5mm) at the detection limit",
        "High resolution (1225×966) captures subtle visual details",
        "Models that do well here truly generalise to hard clinical cases",
    ], color=C.RED)

    h3("How It Differs from CVC-ClinicDB")
    bullet([
        "ETIS-Larib: harder cases, higher resolution, smaller dataset (196 vs 612)",
        "CVC-ClinicDB: standard difficulty, medium resolution (574×500)",
        "Both are used as test benchmarks in most published papers",
        "Performance gap between the two reveals model robustness",
    ])

    warn("196 images — smallest dataset in this project. Never train on this alone")
    good("Use as a challenging test set to evaluate model generalisation")


# ============================================================
# SECTION 6: KID DATASET
# ============================================================

def print_kid_dataset():
    h1("DATASET 5: KID (Capsule Endoscopy Dataset)")

    h2("What Is This Dataset?")
    body("""
KID (Kyoto Intestinal Dataset) is a Wireless Capsule Endoscopy dataset created
at the University of Thessaly, Greece. It contains carefully annotated WCE frames
across 4 GI pathology classes and is primarily used for cross-dataset validation
and model generalisation testing — NOT for primary training.

Note: The full KID dataset Kaggle links (kiddataset/kid-dataset-1 and
kiddataset/kid-dataset-2) are no longer accessible. The 370 images here were
extracted from the KID subfolders inside the Capsule Vision 2024 Challenge dataset,
which are the actual KID images included by the challenge organisers. The KID
authors have been contacted at mdss.uth.gr for the complete dataset.
""")

    h2("Key Characteristics")
    field("Full Name:",    "KID — Kyoto Intestinal Dataset")
    field("Modality:",     "Wireless Capsule Endoscopy (WCE)")
    field("Classes:",      "4 GI pathology classes")
    field("Total Images:", "370 (extracted from Capsule Vision 2024 KID subset)")
    field("Resolution:",   "360 x 360 px")
    field("Purpose:",      "Cross-dataset validation and generalisation testing")
    field("Citation:",     "Koulaouzidis et al., Endoscopy International Open, 2017")

    h2("Class Distribution")
    classes = [
        ("Normal",      315, "Healthy GI mucosa — majority class"),
        ("Polyps",       34, "Abnormal tissue growths — cancer risk"),
        ("Angiectasia",  18, "Abnormal blood vessel formations"),
        ("Bleeding",      3, "Active GI bleeding — critical minority"),
    ]
    total = 370
    for name, count, note in classes:
        minority = count < 20
        class_row(name, count, total, note=note, is_minority=minority)

    print()
    warn("Bleeding has only 3 images — extremely rare even in real WCE examinations")
    warn("KID is NOT used for training — only for cross-dataset evaluation")

    h2("Why KID Matters For This Project")
    bullet([
        "Same WCE modality as Kvasir-Capsule — true cross-dataset WCE test",
        "Independent source — tests model generalisation to unseen WCE data",
        "Small size is intentional — expert-annotated frames only",
        "Standard benchmark used in capsule endoscopy AI research",
    ])

    h2("Class Mapping: KID → Kvasir-Capsule")
    mappings = [
        ("KID: Normal",      "Kvasir: Normal Clean Mucosa", "Same class"),
        ("KID: Angiectasia", "Kvasir: Angiectasia",         "Direct match"),
        ("KID: Bleeding",    "Kvasir: Blood - Fresh",       "Active bleeding"),
        ("KID: Polyps",      "Kvasir: Polyp",               "Direct match"),
    ]
    for kid_cls, kvasir_cls, note in mappings:
        print(f"      {C.CYAN}{kid_cls:<25}{C.WHITE} → {C.GREEN}{kvasir_cls:<30}{C.GRAY} ({note}){C.RESET}")

# SECTION 7: CROSS-DATASET ANALYSIS
# ============================================================

def print_cross_dataset_analysis():
    h1("CROSS-DATASET ANALYSIS")

    h2("Class Overlap Across All 5 Datasets")
    body("""
Several GI findings appear across multiple datasets. Understanding this overlap
is key to building a model that generalises across datasets.
    """)
    print()
    print(f"      {C.CYAN}{C.BOLD}{'GI Finding':<25} {'Kvasir-Cap':>12} {'KVASIR-v2':>11} {'CVC':>8} {'ETIS':>8} {'KID':>8}{C.RESET}")
    print(f"      {'─' * 78}")

    overlaps = [
        ("Polyp",          "55",    "1,000",  "612",  "196",  "302"),
        ("Bleeding",       "446",   "–",      "–",    "–",    "560"),
        ("Ulcer",          "854",   "1,000",  "–",    "–",    "–"),
        ("Angiectasia",    "866",   "–",      "–",    "–",    "1,000"),
        ("Normal mucosa",  "34,338","2,000",  "–",    "–",    "–"),
        ("Inflammation",   "506",   "1,000",  "–",    "–",    "509"),
    ]
    for row in overlaps:
        name = row[0]
        vals = row[1:]
        colored = [f"{C.GREEN if v != '–' else C.GRAY}{v:>8}{C.RESET}" for v in vals]
        print(f"      {C.WHITE}{name:<25}{''.join(colored)}{C.RESET}")

    print()
    info("Green = class exists in that dataset | Gray dash = not present")

    h2("Modality Grouping")
    h3("Group 1: Wireless Capsule Endoscopy (WCE)")
    bullet([
        "Kvasir-Capsule — 47,238 images, 14 classes, extreme imbalance",
        "KID            —  2,371 images,  4 classes, mild imbalance",
        "Both are pill cameras. Images look similar. Same visual domain.",
    ], color=C.BLUE)

    h3("Group 2: Standard Colonoscopy / Gastroscopy")
    bullet([
        "KVASIR v2     — 8,000 images, 8 classes, perfectly balanced",
        "CVC-ClinicDB  —   612 images, 2 classes, perfectly balanced",
        "ETIS-Larib    —   196 images, 2 classes, perfectly balanced",
        "Doctor-controlled scope. Higher quality, different visual domain.",
    ], color=C.MAGENTA)

    h3("Key Challenge for the Model")
    body("""
WCE images and colonoscopy images look different — different camera hardware,
different lighting, different frame rate, different perspective angles. A model
trained only on Kvasir-Capsule may fail on KVASIR v2 and vice versa.

This cross-domain generalisation is a genuine research challenge and part of
what makes this project publication-worthy.
    """)

    h2("Recommended Strategy for Multi-Dataset Training")
    print()
    steps = [
        ("Step 1", "Train foundation model on Kvasir-Capsule (largest, hardest)"),
        ("Step 2", "Fine-tune/validate on KVASIR v2 (balanced, standard endoscopy)"),
        ("Step 3", "Use KID for WCE cross-dataset validation (same modality)"),
        ("Step 4", "Use CVC-ClinicDB and ETIS-Larib for polyp-specific evaluation"),
        ("Step 5", "Apply Knowledge Distillation — teacher on full data, student compressed"),
    ]
    for step, desc in steps:
        print(f"      {C.CYAN}{C.BOLD}{step}:{C.RESET} {C.WHITE}{desc}{C.RESET}")


# ============================================================
# SECTION 8: WHY THIS HAS RESEARCH POTENTIAL
# ============================================================

def print_research_potential():
    h1("WHY THIS PROJECT HAS RESEARCH POTENTIAL")

    h2("What Currently Exists in Literature")
    bullet([
        "Most papers train and test on a SINGLE dataset (usually Kvasir-Capsule alone)",
        "Few papers explicitly handle the 3,434:1 imbalance with proper methodology",
        "Knowledge Distillation for WCE classification is largely unexplored",
        "Cross-dataset generalisation (WCE → Colonoscopy) rarely addressed",
        "Prof. Chandaliya's own paper applied KD to face morphing, not medical imaging",
    ])

    h2("What Your Project Does Differently")
    bullet([
        "Single model compatible with ALL 5 datasets simultaneously",
        "Proper imbalance handling with documented methodology",
        "Knowledge Distillation: compress EfficientNetV2 teacher → lightweight student",
        "Cross-modality evaluation: WCE and colonoscopy in the same framework",
        "Extendable architecture — new datasets can be added without retraining from scratch",
    ], color=C.GREEN)

    h2("Realistic Publication Targets")
    print()
    venues = [
        ("CBMS 2025/2026",
         "IEEE Int. Conf. on Computer-Based Medical Systems",
         "Very achievable — medical AI focus"),
        ("ISBI 2026",
         "IEEE Int. Symposium on Biomedical Imaging",
         "Achievable with strong cross-dataset results"),
        ("MICCAI Workshop",
         "Medical Image Computing and Computer Assisted Intervention",
         "Competitive but possible"),
        ("Computers in Bio & Med",
         "Elsevier Journal (IF ~7.7)",
         "Journal — requires very thorough evaluation"),
    ]
    for venue, full, note in venues:
        print(f"      {C.YELLOW}{C.BOLD}{venue:<20}{C.RESET} {C.WHITE}{full:<45}{C.RESET}")
        print(f"      {C.GRAY}{'':20} → {note}{C.RESET}\n")

    h2("The Core Novel Claim")
    body("""
'We present the first systematic study of cross-dataset generalisation
for gastrointestinal disease classification across both WCE and standard
colonoscopy modalities, combining extreme imbalance handling with
knowledge distillation to produce a compact, deployable model that
performs consistently across all five benchmark datasets.'

That sentence, if supported by results, is publication-worthy.
    """, indent=6)


# ============================================================
# MAIN
# ============================================================

def main():
    print_overview()
    print_wce_explanation()
    print_kvasir_capsule()
    print_kvasir_v2()
    print_cvc_clinicdb()
    print_etis_larib()
    print_kid_dataset()
    print_cross_dataset_analysis()
    print_research_potential()

    h1("SUMMARY — WHAT TO REMEMBER")
    bullet([
        "Kvasir-Capsule is PRIMARY — 47,238 images, 14 classes, 3,434:1 imbalance",
        "KVASIR v2 is the BALANCED BENCHMARK — 8,000 images, 1,000 per class",
        "CVC-ClinicDB and ETIS-Larib are POLYP-SPECIFIC — use for evaluation",
        "KID is SAME MODALITY as Kvasir-Capsule — use for WCE cross-validation",
        "WCE ≠ Colonoscopy — different visual domains, model must generalise both",
        "Minority classes = most critical clinically — polyps, bleeding, ulcers",
        "Standard accuracy is meaningless here — use F1-score and recall per class",
        "Knowledge Distillation compresses the model for real clinical deployment",
    ], color=C.CYAN)

    print(f"\n{C.GREEN}{C.BOLD}  Run task1_exploration.py to generate all charts and visualisations.{C.RESET}\n")


if __name__ == '__main__':
    main()


════════════════════════════════════════════════════════════════════════
  DATASET OVERVIEW — WCE Gastrointestinal Classification
════════════════════════════════════════════════════════════════════════
    This project works with 5 medical imaging datasets for gastrointestinal (GI)
    disease classification. Together they cover two distinct imaging modalities:
    
    (1) Wireless Capsule Endoscopy (WCE) — patient swallows a pill-sized camera
    (2) Standard Colonoscopy — a flexible tube inserted by a gastroenterologist
    
    Understanding both modalities is essential because the project requires a
    single model that generalises across all 5 datasets.

  ◆ The 5 Datasets At A Glance

      Dataset                Modality                Classes     Images Imbalance         
      ────────────────────────────────────────────────────────────────────────────────
      Kvasir-Capsule         WCE (Capsule)                14     47,238  EXTREME  3434:1
      KVASIR v2              